# Graph vs Relational

## Overview
This notebook compares graph and relational approaches to the same problem, shows when each wins, and covers hybrid architectures.

**The key insight:**

> Every additional hop in a traversal costs one JOIN in SQL but one pointer follow in a graph database. At 1-2 hops, SQL wins on familiarity. At 5+ variable hops, the graph wins decisively.

**The core trade-off:**

| | Graph (Neo4j) | Relational (PostgreSQL) |
|---|---|---|
| Variable-depth traversal | Native, O(1) per hop | Recursive CTE; awkward; can be slow |
| Bulk aggregation | Possible but not primary | Primary strength |
| Schema flexibility | High; nodes can differ | Low; schema change = ALTER TABLE |
| Operational simplicity | Separate server required | Already in your stack |
| SQL familiarity | Requires learning Cypher | Universally known |

---

In [1]:
import sqlite3, networkx as nx

print("=== Modelling the same problem: relational vs graph ===")
print()

# The problem: clinical research network
# Who are the co-investigators of my direct collaborators' trials?

print("Problem: find 2nd-degree trial connections for Aroha Ngata")
print("  (trials that her direct collaborators are leading/contributing to)")
print()

# ── Relational approach ───────────────────────────────────────────
print("RELATIONAL approach (SQL):")
sql_approach = """
-- 3-join query to find 2nd-degree trial connections
SELECT DISTINCT t2.title AS trial_title, r2.name AS colleague
FROM   researchers r1
JOIN   researcher_trials rt1  ON r1.id = rt1.researcher_id
JOIN   researcher_trials rt2  ON rt1.researcher_id != rt2.researcher_id
                              AND rt1.trial_id = rt2.trial_id  -- same trial = colleague
JOIN   researchers r2         ON rt2.researcher_id = r2.id
JOIN   researcher_trials rt3  ON r2.id = rt3.researcher_id     -- colleague's other trials
JOIN   trials t2              ON rt3.trial_id = t2.id
WHERE  r1.name = 'Aroha Ngata'
  AND  t2.id NOT IN (
         SELECT trial_id FROM researcher_trials WHERE researcher_id = r1.id
       )
ORDER BY trial_title;
-- Adding more hops: each additional hop adds 2 more JOINs
"""
print(sql_approach)

print("GRAPH approach (Cypher):")
cypher_approach = """
-- Exact same query — 2 pattern hops
MATCH (r1:Researcher {name: 'Aroha Ngata'})
      -[:LEADS|CONTRIBUTES_TO]->(:Trial)
      <-[:LEADS|CONTRIBUTES_TO]-(r2:Researcher)
      -[:LEADS|CONTRIBUTES_TO]->(t2:Trial)
WHERE NOT (r1)-[:LEADS|CONTRIBUTES_TO]->(t2)
RETURN DISTINCT t2.title AS trial_title, r2.name AS colleague
ORDER BY trial_title

-- Adding more hops: just change *1 to *2 or *3
MATCH (r1:Researcher {name: 'Aroha Ngata'})
      -[:LEADS|CONTRIBUTES_TO*1..3]->(:Trial)  -- variable depth, one change
      ...
"""
print(cypher_approach)


=== Modelling the same problem: relational vs graph ===

Problem: find 2nd-degree trial connections for Aroha Ngata
  (trials that her direct collaborators are leading/contributing to)

RELATIONAL approach (SQL):

-- 3-join query to find 2nd-degree trial connections
SELECT DISTINCT t2.title AS trial_title, r2.name AS colleague
FROM   researchers r1
JOIN   researcher_trials rt1  ON r1.id = rt1.researcher_id
JOIN   researcher_trials rt2  ON rt1.researcher_id != rt2.researcher_id
                              AND rt1.trial_id = rt2.trial_id  -- same trial = colleague
JOIN   researchers r2         ON rt2.researcher_id = r2.id
JOIN   researcher_trials rt3  ON r2.id = rt3.researcher_id     -- colleague's other trials
JOIN   trials t2              ON rt3.trial_id = t2.id
WHERE  r1.name = 'Aroha Ngata'
  AND  t2.id NOT IN (
         SELECT trial_id FROM researcher_trials WHERE researcher_id = r1.id
       )
ORDER BY trial_title;
-- Adding more hops: each additional hop adds 2 more JOINs

GRAPH

---
## Live SQL vs graph comparison

In [2]:
import sqlite3, networkx as nx

print("=== Live comparison: relational vs graph query ===")
print()

# ── Build relational schema ───────────────────────────────────────
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row
conn.executescript("""
CREATE TABLE researchers   (id TEXT PRIMARY KEY, name TEXT, role TEXT);
CREATE TABLE trials        (id TEXT PRIMARY KEY, title TEXT, phase INTEGER);
CREATE TABLE researcher_trials (researcher_id TEXT, trial_id TEXT, contribution TEXT);
CREATE TABLE collaborations    (r1_id TEXT, r2_id TEXT);

INSERT INTO researchers VALUES
  ('R001','Aroha Ngata','PI'),('R002','Liam Chen','Biostatistician'),
  ('R003','Fatima Rashid','Coordinator'),('R004','James MacLeod','Researcher');
INSERT INTO trials VALUES
  ('T001','Cardio Alpha',2),('T002','Neuro Beta',3),('T003','Oncology Gamma',1);
INSERT INTO researcher_trials VALUES
  ('R001','T001','LEADS'),('R002','T001','CONTRIBUTES_TO'),('R003','T001','CONTRIBUTES_TO'),
  ('R004','T002','LEADS'),('R002','T002','CONTRIBUTES_TO'),('R003','T003','CONTRIBUTES_TO');
INSERT INTO collaborations VALUES ('R001','R004'),('R002','R005'),('R001','R003');
""")
conn.commit()

# SQL: 2-hop colleague trials
print("SQL: trials led by Aroha's colleagues (2-hop)")
sql_rows = conn.execute("""
  SELECT DISTINCT t2.title, r2.name AS colleague
  FROM   researcher_trials rt1
  JOIN   researchers r1 ON r1.id = rt1.researcher_id AND r1.name = 'Aroha Ngata'
  JOIN   researcher_trials rt2
         ON  rt2.trial_id = rt1.trial_id
         AND rt2.researcher_id != r1.id
  JOIN   researchers r2 ON r2.id = rt2.researcher_id
  JOIN   researcher_trials rt3 ON rt3.researcher_id = r2.id
  JOIN   trials t2 ON t2.id = rt3.trial_id
  WHERE  rt3.trial_id NOT IN (
           SELECT trial_id FROM researcher_trials WHERE researcher_id = r1.id
         )
  ORDER BY t2.title
""").fetchall()
for r in sql_rows:
    print(f"  {r['title']:<22s}  via colleague: {r['colleague']}")

print()

# Graph: same query
G = nx.MultiDiGraph()
for rid, name in [("R001","Aroha Ngata"),("R002","Liam Chen"),
                  ("R003","Fatima Rashid"),("R004","James MacLeod")]:
    G.add_node(rid, label="Researcher", name=name)
for tid, title in [("T001","Cardio Alpha"),("T002","Neuro Beta"),("T003","Oncology Gamma")]:
    G.add_node(tid, label="Trial", title=title)
for s, d, r in [
    ("R001","T001","LEADS"),("R002","T001","CONTRIBUTES_TO"),("R003","T001","CONTRIBUTES_TO"),
    ("R004","T002","LEADS"),("R002","T002","CONTRIBUTES_TO"),("R003","T003","CONTRIBUTES_TO"),
]:
    G.add_edge(s, d, rel_type=r)

print("Graph: same 2-hop query (networkx traversal)")
aroha_trials = {v for u, v, ed in G.edges(data=True)
                if u == "R001" and ed['rel_type'] in ('LEADS','CONTRIBUTES_TO')}
colleagues = set()
for t in aroha_trials:
    for u, v, ed in G.edges(data=True):
        if v == t and u != "R001" and ed['rel_type'] in ('LEADS','CONTRIBUTES_TO'):
            colleagues.add(u)

colleague_trials = {}
for c in colleagues:
    for u, v, ed in G.edges(data=True):
        if u == c and ed['rel_type'] in ('LEADS','CONTRIBUTES_TO') and v not in aroha_trials:
            cname = G.nodes[c]['name']
            ttitle = G.nodes[v]['title']
            colleague_trials[ttitle] = cname

for ttitle, cname in sorted(colleague_trials.items()):
    print(f"  {ttitle:<22s}  via colleague: {cname}")


=== Live comparison: relational vs graph query ===

SQL: trials led by Aroha's colleagues (2-hop)
  Neuro Beta              via colleague: Liam Chen
  Oncology Gamma          via colleague: Fatima Rashid

Graph: same 2-hop query (networkx traversal)
  Neuro Beta              via colleague: Liam Chen
  Oncology Gamma          via colleague: Fatima Rashid


---
## Decision framework and hybrid architectures

In [3]:
print("=== Decision framework: graph vs relational ===")
print()

decision_matrix = [
    ("Data structure",
     "Highly connected; relationships are first-class",
     "Tabular; entities have fixed attributes"),
    ("Query pattern",
     "Traversal: 'who knows who knows who'; paths",
     "Aggregation: counts, sums, averages by category"),
    ("Depth of traversal",
     "Variable depth (1..N hops) is the main access pattern",
     "Fixed depth; extra hops = extra JOINs (awkward)"),
    ("Schema flexibility",
     "Nodes can have different properties; schema evolves easily",
     "Schema changes require ALTER TABLE; rigid structure"),
    ("Many-to-many",
     "Direct edges in both directions; no junction table",
     "Junction table required; 3-table JOIN for each hop"),
    ("Data volume",
     "Works up to billions of nodes/edges; optimised for traversal",
     "Scales to petabytes; optimised for scan + aggregate"),
    ("Team familiarity",
     "Requires Cypher/graph thinking; steeper learning curve",
     "SQL is universally known; easier to hire and onboard"),
]

print(f"  {'Factor':<22s}  {'Choose Graph when...':<42s}  Choose Relational when...")
print("  " + "-"*100)
for factor, graph_when, rel_when in decision_matrix:
    print(f"  {factor:<22s}  {graph_when:<42s}  {rel_when}")

print()
print("Hybrid architectures (very common in practice):")
hybrids = [
    ("PostgreSQL primary + Neo4j graph layer",
     "Transactional data in Postgres; graph analytics on a synced Neo4j replica"),
    ("PostgreSQL JSONB + recursive CTE",
     "Graph-like queries on small-medium connected data without a separate graph DB"),
    ("Apache AGE (PostgreSQL extension)",
     "Native Cypher queries inside PostgreSQL — graph + SQL in one database"),
    ("DuckDB + networkx",
     "Analytical SQL for aggregation; networkx for graph algorithms in Python"),
]
for arch, desc in hybrids:
    print(f"  {arch}")
    print(f"    {desc}")
    print()

print("Apache AGE example (graph in PostgreSQL):")
print("""
  -- Enable AGE extension:
  CREATE EXTENSION age;
  SET search_path = ag_catalog, public;

  -- Create a graph:
  SELECT create_graph('research_graph');

  -- Load data and run Cypher inside PostgreSQL:
  SELECT * FROM cypher('research_graph', $$
      MATCH (r:Researcher)-[:LEADS]->(t:Trial)
      RETURN r.name, t.title
  $$) AS (researcher agtype, trial agtype);
""")


=== Decision framework: graph vs relational ===

  Factor                  Choose Graph when...                        Choose Relational when...
  ----------------------------------------------------------------------------------------------------
  Data structure          Highly connected; relationships are first-class  Tabular; entities have fixed attributes
  Query pattern           Traversal: 'who knows who knows who'; paths  Aggregation: counts, sums, averages by category
  Depth of traversal      Variable depth (1..N hops) is the main access pattern  Fixed depth; extra hops = extra JOINs (awkward)
  Schema flexibility      Nodes can have different properties; schema evolves easily  Schema changes require ALTER TABLE; rigid structure
  Many-to-many            Direct edges in both directions; no junction table  Junction table required; 3-table JOIN for each hop
  Data volume             Works up to billions of nodes/edges; optimised for traversal  Scales to petabytes; optimised for

---
## Common Pitfalls

**1. Choosing graph for simple tabular reporting**
If your primary queries are `COUNT`, `AVG`, `SUM GROUP BY`, and `JOIN` across well-defined entities, a graph database adds operational complexity with no benefit. Use PostgreSQL.

**2. Choosing graph because the data 'looks like a graph'**
All relational data can be visualised as a graph. The question is whether your queries require multi-hop traversal, path finding, or variable-depth exploration. If yes, graph is appropriate. If no, relational is simpler.

**3. Not modelling bidirectional relationships explicitly**
Neo4j relationships are directed. For undirected queries, you can use `-[:TYPE]-` (no arrow). But for storage efficiency and semantic clarity, decide on canonical direction and query consistently. Storing both `(A)-[:KNOWS]->(B)` and `(B)-[:KNOWS]->(A)` doubles storage and can cause double-counting in aggregations.

**4. Ignoring operational complexity of running Neo4j**
Neo4j requires a separate database server, Java runtime, its own backup process, memory tuning (heap + page cache), and monitoring. PostgreSQL is already running in most stacks. Factor operational cost into the graph vs relational decision.

**5. Missing the Apache AGE option**
Apache AGE brings native Cypher graph queries into PostgreSQL. For teams already running PostgreSQL who need graph querying on moderate-sized data, AGE may be the right choice — one database, SQL + Cypher, no separate Neo4j instance to run.

**6. Not indexing node properties used in MERGE and MATCH WHERE**
In both Neo4j and relational databases, properties used in frequent lookups must be indexed. In Neo4j: `CREATE INDEX researcher_id FOR (r:Researcher) ON (r.id)`. Without indexes, every MATCH with a WHERE clause does a full label scan.


---
*sql_methods_library - Samantha McGarrigle*